# Cross-Day Partial Unwinds Report

**Period:** March 1-10, 2026  
**Source:** DTCC SDR USD SOFR/FF swaps

In [1]:
import nest_asyncio
nest_asyncio.apply()

import sys, os

_nb_dir = os.path.dirname(os.path.abspath("__file__"))
_project_root = os.path.normpath(os.path.join(_nb_dir, "..", ".."))
for p in [_nb_dir, _project_root]:
    if p not in sys.path:
        sys.path.insert(0, p)

import datetime
import pandas as pd
import _usd_swaps_common as sdr

sdr.notebook_setup()

In [7]:
from SDRUtils.analytics.trade_tape import TradeTape

df = sdr.load_usd_swaps(
    datetime.datetime(2026, 5, 18),
    datetime.datetime(2026, 5, 22),
)
tape = TradeTape(df)
enriched = tape.compute()
print(f"Raw: {len(df):,} trades | Enriched: {len(enriched.columns)} columns")
tape.summary()

MERGING SLICES...: 100%|██████████| 5/5 [00:00<00:00, 35.17it/s]


TradeTape:   0%|          | 0/16 [00:00<?, ?layer/s]

Raw: 20,476 trades | Enriched: 149 columns


{'n_trades': 20476,
 'n_new_risk': 20473,
 'pct_new_risk': 100.0,
 'pct_compression': 0.0,
 'pct_ufro': np.float64(44.5),
 'pct_block': np.float64(2.9),
 'pct_capped': np.float64(2.4),
 'top_trade_types': {'OUTRIGHT': 8075,
  'CURVE': 3126,
  'IMM': 2944,
  'FLY': 1839,
  'MATCHED_MATURITY': 927},
 'venue_split': {'D2C': 18694, 'D2D': 1782},
 'ccp_split': {'LCH': 20476},
 'n_xd_terminated': 0,
 'n_xd_partial_unwind': 0,
 'n_novations': 0,
 'n_compressions_spec': 0,
 'n_exercise_born': 0,
 'n_clearing_terminations': 0,
 'n_non_standard_term': 382}

## Cross-Day Lifecycle Resolution

Now load with `raw_df` to enable cross-day lifecycle resolution.

In [8]:
from SDRUtils.core.lifecycle import resolve_lifecycle_cross_day

classified_df, raw_df = sdr.load_usd_swaps(
    datetime.datetime(2026, 3, 1),
    datetime.datetime(2026, 3, 11),  # extend to capture full Mar 10 events
    return_raw=True,
)
print(f"Classified: {len(classified_df):,} | Raw (unfiltered): {len(raw_df):,}")

MERGING SLICES...: 100%|██████████| 9/9 [00:00<00:00, 16.74it/s]


  [WARN] Classification failed for 2026-03-08: 2026-03-08 is not a business day in the US government bond market calendar!
  [WARN] Classification failed for 2026-03-07: 2026-03-07 is not a business day in the US government bond market calendar!
Classified: 37,982 | Raw (unfiltered): 253,907


In [9]:
# Run cross-day resolver on all classified trades
classified_ids = set(classified_df["trade_id"].astype(str))
xd = resolve_lifecycle_cross_day(raw_df, classified_ids, skip_intraday_only=True)
print(f"Cross-day resolved: {len(xd):,} trades")
print()
print("Status distribution:")
print(xd["xd_status"].value_counts())

Cross-day resolved: 296 trades

Status distribution:
xd_status
ACTIVE            156
TERMINATED        122
ERRORED            16
PARTIAL_UNWIND      2
Name: count, dtype: int64


## Partial Unwinds

In [10]:
# Filter to partial unwinds
partial = xd[xd["xd_has_partial_unwind"] == True].copy()
print(f"Partial unwinds: {len(partial)}")
print()

if not partial.empty:
    # Merge with classified data for context
    partial.index.name = "trade_id"
    partial_merged = partial.reset_index().merge(
        classified_df[["trade_id", "tenor_label", "fixed_rate", "notional", "product_type", "execution_timestamp"]],
        on="trade_id",
        how="left",
    )
    
    display_cols = [
        "trade_id", "tenor_label", "product_type",
        "xd_inception_notional", "xd_current_notional", 
        "xd_notional_pct_remaining", "xd_status",
        "xd_n_events", "xd_n_days_spanned",
        "xd_is_terminated", "xd_fields_changed",
    ]
    available = [c for c in display_cols if c in partial_merged.columns]
    
    print("Partial unwind details:")
    display(partial_merged[available].sort_values("xd_notional_pct_remaining"))
else:
    print("No partial unwinds found in this date range.")

Partial unwinds: 3

Partial unwind details:


,trade_id,tenor_label,product_type,xd_inception_notional,xd_current_notional,xd_notional_pct_remaining,xd_status,xd_n_events,xd_n_days_spanned,xd_is_terminated,xd_fields_changed
2,2304676889000000101,10Y,OIS_SWAP,25000000.0,5.0,2.000000e-07,TERMINATED,22,2,True,"Notional amount-Leg 1,Notional amount-Leg 2"
1,2235384196000000201,5Y,OIS_SWAP,6000000.0,5.0,8.333333e-07,PARTIAL_UNWIND,7,3,False,"Notional amount-Leg 1,Notional amount-Leg 2,Ot..."
0,2217177190000000101,10Y,OIS_SWAP,1000000.0,5.0,5.000000e-06,PARTIAL_UNWIND,7,4,False,"Notional amount-Leg 1,Notional amount-Leg 2,Ot..."


In [11]:
# Summary statistics
if not partial.empty:
    print("=== Partial Unwind Summary (Mar 1-10, 2026) ===")
    print(f"Total partial unwinds: {len(partial)}")
    print(f"  Still active: {(partial['xd_status'] == 'PARTIAL_UNWIND').sum()}")
    print(f"  Subsequently terminated: {(partial['xd_status'] == 'TERMINATED').sum()}")
    print()
    print(f"Notional remaining distribution:")
    print(partial["xd_notional_pct_remaining"].describe())
    print()
    print(f"Days spanned distribution:")
    print(partial["xd_n_days_spanned"].value_counts().sort_index())
    print()
    print(f"Events per trade distribution:")
    print(partial["xd_n_events"].describe())

=== Partial Unwind Summary (Mar 1-10, 2026) ===
Total partial unwinds: 3
  Still active: 2
  Subsequently terminated: 1

Notional remaining distribution:
count    3.000000e+00
mean     2.011111e-06
std      2.607752e-06
min      2.000000e-07
25%      5.166667e-07
50%      8.333333e-07
75%      2.916667e-06
max      5.000000e-06
Name: xd_notional_pct_remaining, dtype: float64

Days spanned distribution:
xd_n_days_spanned
2    1
3    1
4    1
Name: count, dtype: int64

Events per trade distribution:
count     3.000000
mean     12.000000
std       8.660254
min       7.000000
25%       7.000000
50%       7.000000
75%      14.500000
max      22.000000
Name: xd_n_events, dtype: float64
